# 04 — Respiratory Conditions Q&A: RAG Pipeline

**Component 4 of 4 — RespiraAI Deployment Capstone**

Retrieval-augmented generation over 12 short respiratory-condition write-ups (pneumonia, lung cancer, COPD, asthma, TB, bronchitis, pulmonary embolism, sleep apnea, pulmonary fibrosis, COVID-19, pleurisy, cystic fibrosis — written from general medical knowledge, not copied from any single source).

**Retrieval uses TF-IDF, not dense embeddings** — a deliberate choice, not a shortcut taken only because of this environment: it needs zero model downloads (no `sentence-transformers` + PyTorch dependency, no Hugging Face Hub call at runtime), keeps the deployed Docker image small, and for a 12-document knowledge base with fairly distinct topics, keyword-overlap retrieval is *plenty* good — you don't need semantic embeddings to tell a sleep-apnea question from a lung-cancer question. If you later grow the knowledge base to hundreds of documents with more topic overlap, that's the point to revisit and swap in `sentence-transformers` + FAISS/Chroma; the retrieval interface below (`retrieve(query, k)`) is written so that swap only touches this one function.


In [1]:
import os
import glob
import json
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI


## Load the knowledge base

In [2]:
def find_docs_dir():
    candidates = [
        os.path.join("data", "rag_docs"),
        os.path.join("day1_data", "data", "rag_docs"),
        os.path.join("..", "data", "rag_docs"),
        "rag_docs",
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"Could not find rag_docs/ in any of: {candidates}")

DOCS_DIR = find_docs_dir()
doc_paths = sorted(glob.glob(os.path.join(DOCS_DIR, "*.txt")))

doc_names, doc_texts = [], []
for path in doc_paths:
    with open(path, encoding="utf-8") as f:
        doc_texts.append(f.read())
    doc_names.append(os.path.splitext(os.path.basename(path))[0])

print(f"Loaded {len(doc_texts)} documents: {doc_names}")


Loaded 12 documents: ['asthma', 'bronchitis', 'copd', 'covid19_respiratory', 'cystic_fibrosis', 'lung_cancer', 'pleurisy', 'pneumonia', 'pulmonary_embolism', 'pulmonary_fibrosis', 'sleep_apnea', 'tuberculosis']


## Build the TF-IDF index

`TfidfVectorizer` turns each document into a sparse vector where each dimension is a word, weighted by how distinctive that word is to this document relative to the whole collection (term frequency x inverse document frequency). A question gets embedded into the same space, and cosine similarity ranks documents by word-overlap relevance.

In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(doc_texts)
print("vocabulary size:", len(vectorizer.vocabulary_))
print("doc-term matrix shape:", doc_vectors.shape)


def retrieve(query, k=3):
    """Return the top-k (doc_name, text, score) tuples most relevant to query."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_vectors)[0]
    top_idx = scores.argsort()[::-1][:k]
    return [(doc_names[i], doc_texts[i], float(scores[i])) for i in top_idx]


vocabulary size: 777
doc-term matrix shape: (12, 777)


## Test retrieval quality

Worth checking retrieval *before* wiring in the LLM — if the wrong documents get retrieved, a good LLM prompt can't fix that; garbage in, garbage out applies to RAG more than almost anywhere else.

In [4]:
test_queries = [
    "What causes shortness of breath during sleep?",
    "Is bacterial pneumonia treated differently than viral pneumonia?",
    "What are the symptoms of lung cancer?",
    "How is a blood clot in the lung treated?",
]

for q in test_queries:
    print(f"\nQ: {q}")
    for name, _, score in retrieve(q, k=3):
        print(f"  {name:<22} score={score:.3f}")



Q: What causes shortness of breath during sleep?
  sleep_apnea            score=0.415
  pleurisy               score=0.105
  pulmonary_fibrosis     score=0.091

Q: Is bacterial pneumonia treated differently than viral pneumonia?
  pneumonia              score=0.422
  pleurisy               score=0.087
  covid19_respiratory    score=0.061

Q: What are the symptoms of lung cancer?
  lung_cancer            score=0.483
  pulmonary_fibrosis     score=0.101
  pleurisy               score=0.100

Q: How is a blood clot in the lung treated?
  pulmonary_embolism     score=0.321
  pneumonia              score=0.164
  lung_cancer            score=0.091


## Wire in the LLM

Same Groq/GPT-OSS setup as the prompt-engineering lab — `openai/gpt-oss-20b`, `reasoning_effort="low"` to keep the internal reasoning trace from eating the whole `max_tokens` budget before it writes a visible answer, same `GROQ_API_KEY` environment variable pattern.

In [5]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY or "unset")
MODEL_NAME = "openai/gpt-oss-20b"


def call_llm(messages, temperature=0.3, max_tokens=400):
    if not GROQ_API_KEY:
        raise RuntimeError("GROQ_API_KEY is not set; configure it in the environment before calling the LLM.")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


## The RAG function

Retrieve top-k chunks, build a prompt that (a) instructs the model to answer *only* from the provided context and (b) say so if the context doesn't cover the question — same grounding discipline as the document Q&A exercise from the prompt-engineering lab, applied here with retrieved context instead of one hand-pasted document. Returns the answer plus which source documents it drew from, so a frontend can show provenance rather than asking the user to trust an unsourced answer.

In [6]:
def ask_rag(question, k=3):
    retrieved = retrieve(question, k=k)

    context_block = "\n\n".join(
        f"[Source: {name}]\n{text}" for name, text, _score in retrieved
    )

    prompt = f"""Answer the question using ONLY the information in the sources below.
If the sources don't contain enough information to answer, say so honestly rather than guessing.
Keep the answer concise (3-5 sentences) and do not invent medical claims not present in the sources.

Sources:
{context_block}

Question: {question}

Answer:"""

    answer = call_llm([{"role": "user", "content": prompt}])
    sources = [name for name, _, _ in retrieved]
    return {"answer": answer, "sources": sources}


In [7]:
# Test with a couple of real questions when the Groq key is configured.
if GROQ_API_KEY:
    for q in [
        "What's the difference between asthma and COPD?",
        "When should someone with pneumonia symptoms see a doctor urgently?",
        "Can vitamin C cure a common cold?",  # deliberately off-topic -- checks the model says so rather than guessing
    ]:
        print(f"\nQ: {q}")
        result = ask_rag(q)
        print(f"A: {result['answer']}")
        print(f"Sources: {result['sources']}")
else:
    print("Skipping live LLM questions -- GROQ_API_KEY is not set.")


Skipping live LLM questions -- GROQ_API_KEY is not set.


## Persist the index

So the FastAPI app can load a pre-built index at startup instead of re-vectorizing all 12 documents on every container start (cheap here, but the same principle matters more once the knowledge base grows).

In [8]:
RAG_STORE_DIR = "../app/rag_store" if os.path.isdir("../app") else "rag_store"
os.makedirs(RAG_STORE_DIR, exist_ok=True)

joblib.dump(vectorizer, os.path.join(RAG_STORE_DIR, "tfidf_vectorizer.pkl"))
joblib.dump(doc_vectors, os.path.join(RAG_STORE_DIR, "doc_vectors.pkl"))
with open(os.path.join(RAG_STORE_DIR, "doc_metadata.json"), "w") as f:
    json.dump({"names": doc_names, "texts": doc_texts}, f)

print(f"Saved TF-IDF index to {RAG_STORE_DIR}/")


Saved TF-IDF index to rag_store/


## Notes

- The off-topic test question ("vitamin C cure a common cold") is there on purpose — it's the single most important thing to verify in any RAG system before trusting it: does it correctly decline to answer from thin air when the knowledge base genuinely doesn't cover something, or does it quietly fall back on the underlying LLM's general knowledge and blur the line between "grounded in my sources" and "the model just knows this"? Check your own run's answer to that one specifically.
- TF-IDF retrieval quality depends entirely on shared vocabulary between the question and the documents — if a user asks about a symptom using different words than the documents use, retrieval can miss even when the document logically covers it. This is the concrete tradeoff against dense embeddings, which would catch that kind of paraphrase; worth watching for in your own testing.
- All 4 components are now built: classical ML + DL both solving the lung-cancer task (Notebooks 1-2), CV on chest X-rays (Notebook 3), and this RAG pipeline. Day 2 wires all four into one FastAPI backend.